## Importaciones y constantes

In [ ]:
!pip install -q chromadb
!pip install  -U -q "google-genai"
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q wikipedia

print("Instalado correctamente")

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

load_dotenv()

# Interfaz de LangChain para usar la función de embedding de Gemini
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Ruta en la que se guardarán las colecciones de Chroma
PERSIST_DIRECTORY="./chroma_db"

# Clientes de ChromaDB usando la interfaz de LangChain
# uno por cada colección (movies, people y reviews)

# vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

## Consultas a la API de TMDB

In [ ]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Consultas a Wikipedia

In [ ]:
import wikipedia

def search_in_wikipedia(query: str, isMovie: bool = False):
    if isMovie:
        query += " (film)"

    try:
        summary = wikipedia.summary(query)
        url = wikipedia.page(query).url
        content = wikipedia.page(query).content

        return {"page_content": summary + content, "url": url}
    except wikipedia.exceptions.DisambiguationError as e:
        # Si hay ambigüedad, elegir la opción que contenga "film" o "película"
        for option in e.options:
            if "film" in option.lower() or "película" in option.lower():
                summary = wikipedia.summary(option)
                url = wikipedia.page(option).url
                content = wikipedia.page(query).content

                return {"page_content": summary + content, "url": url}
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}
    except Exception:
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}

## Añadir películas a la colección "movies" de ChromaDB

In [ ]:
def add_movies_to_collection(title: str):
  """
  Busca infomación de una película a partir de su título.

  Args:
    title (str): El título de la película sobre la que buscamos información.
  """
  movies_info = get_movies_info(title)

  # Vector Store de la base de datos de ChromaDB
  vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  print("Added these movies to collection ('movies'):")

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = ""
    cast = ""

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast
  
  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "release_year": movie['release_date'][:4],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count'],
    }
  
  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = vector_store_movies.get(
      ids=[str(movie['id'])],
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:

      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      # Añadimos el documento con la información de la peli
      docsToAdd.append(Document(
        page_content=movie['overview'],
        metadata=get_metadata(movie, director, cast),
        id=str(movie['id'])
      ))

      print(f"Added {movie['title']}.")

  if len(docsToAdd) > 0:
    vector_store_movies.add_documents(docsToAdd)

  print(f"Added {len(docsToAdd)} movies.")


## Añadir reviews a la colección "reviews" de ChromaDB

In [ ]:
def add_reviews_to_collection(title: str):
  """
  Busca reviews y opiniones sobre una película en concreto.

  Args:
    title (str): El título de la película sobre la que necesitamos opiniones.
  """

  print(f"Added these reviews for the movie '{title}' to collection ('reviews'):")

  movies = get_movies_info(title)
  for movie in movies['results']:
    add_reviews_from_id(movie['id'], movie['title'])
    

def add_reviews_from_id(movie_id, movie_title):  
  vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  # Consultamos con nuestra colección a través
  # del id de la película
  result = vector_store_reviews.get(
    where={"movie_id": movie_id},
  )

  # Si NO hay reviews para esa película,
  # se intentan añadir
  if (len(result['ids']) == 0):
    reviews = get_movie_reviews(movie_id)

    def get_metadata(review):
      if review["author_details"]["rating"]:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
            "rating"  : review["author_details"]["rating"]
        }
      else:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
        }

    for review in reviews['results']:

      if review['content']:
        docsToAdd.append(Document(
          page_content=review['content'],
          metadata=get_metadata(review),
          id=str(review['id'])
        ))

    if len(docsToAdd) > 0:
      vector_store_reviews.add_documents(docsToAdd)

    print(f"Reviews for '{movie_title}': Added {len(docsToAdd)} reviews.")
  else:
    print(f"Reviews for '{movie_title}' already in database.")


## Añadir actores a la colección "people" de ChromaDB

In [ ]:
import json
import requests

def add_person_to_collection(name: str):
  """
  Busca información de una persona del cast de una película.

  Args:
    name (str): El nombre de la persona que el usuario está buscando.
  """
  # Buscamos los el nombre para encontrar ilos d
  url = f"https://api.themoviedb.org/3/search/person?query={name}&include_adult=false&language=en-US&page=1"
  r = requests.get(url, headers=TMDB_HEADERS)
  response = json.loads(r.text)

  print("Added these people to collection ('people'): ")

  for person in response['results']:
    add_person_from_id(person['id'])

def add_person_from_id(person_id):
  vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  # por si ya hemos guardado a esa persona
  result = vector_store_people.get(
    ids=[str(details['id'])]
  )

  if not result['ids']:
    # Como TMDB no tiene mucha información sobre actores,
    # usamos wikipedia para obtener el contenido.
    wikiRes=search_in_wikipedia(details['name'])
    content = wikiRes['page_content']

    if not content:
      if details['biography']:
        content = details['biography']
      else:
        content = details['name']

    # Añadir género de la persona
    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    # add_documents pide una lista, así que creamos una
    # con la persona que vamos a añadir
    docsToAdd.append(Document(
      page_content=content,
      metadata={'name': details['name'], 'department': details['known_for_department'], 'gender': gender},
      id=str(details['id'])
    ))
    vector_store_people.add_documents(docsToAdd)

    print(f"Added {details['name']}")
  else:
    print(f"{details['name']} already exists in collection")



## Preguntar a la base de datos vectorial

In [20]:
def query_col(query: str, collection: str, release_year : int | str | None = None):
    """
    Consulta a una colección de las disponibles (movies, reviews y people)
    con la pregunta que ha hecho el usuario, para responder con información veraz.

    Args:
        query (str): La pregunta del usuario.
        collection (str): La colección a consultar: 'movies', 'reviews' o 'people'.
        release_year (int | str | None): Un filtro para ver solo las películas salidas en un año en específico.
    """

    # Cogemos la colección dependiendo de lo que necesitemos
    vector_store = Chroma(collection_name=collection, embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

    # Similarity search hace una búsqueda utilizando la función de embeddings del vector_store
    if not release_year:
        results = vector_store.similarity_search(
            query=query,
            k=10,
        )
    else:
        results = vector_store.similarity_search(
            query=query,
            k=10,
            filter={'release_year': str(release_year)}
        )

    # Lo que se devolverá, un array de diccionarios que tendrán dos propiedades:
    # - page_content (str): Los documentos en Chroma, el texto.
    # - metadata (dict): Metadata de los resultados (título, votos...).
    formatted_results: list[dict] = []

    for doc in results:
        formatted_results.append({"page_content": doc.page_content, "metadata": doc.metadata})

    # Si estamos consultando la colección 'movies', filtramos y ordenamos
    if collection == "movies":
        # Filtrar solo las películas con más de 100 votos
        formatted_results = [r for r in formatted_results if r["metadata"].get("vote_count", 0) > 100]
        # Ordenar por vote_average descendente
        formatted_results.sort(key=lambda x: x["metadata"].get("vote_average", 0), reverse=True)
                # Filtrar por coincidencia exacta de título
        exact_matches = [r for r in formatted_results if query.lower() in r["metadata"].get("movie_title", "").lower()]

        if not exact_matches:
            return "Not Found"
        return exact_matches

    return formatted_results

---
<h2>Añadir y probar la BD</h2>

In [28]:
# Añadir películas
add_movies_to_collection("Minecraft")

# Añadir reviews de la película
add_reviews_to_collection("Inception")

# Añadir actores principales
add_person_to_collection("Leonardo DiCaprio")


Added these movies to collection ('movies'):
Added A Minecraft Movie.
Added Minecraft: The Story of Mojang.
Added A Minecraft Movie 2.
Added Minecraft: Through the Nether Portal.
Added Minecraft: Into the Nether.
Added Minecraft: The Story of Minecraft.
Added 6 movies.
Added these reviews for the movie 'Inception' to collection ('reviews'):
Reviews for 'Inception' already in database.
Reviews for 'The Crack: Inception': Added 0 reviews.
Reviews for 'Syndrome Halla, the Inception of Croatian Professional Film – Born to Die': Added 0 reviews.
Reviews for 'WWA The Inception': Added 0 reviews.
Reviews for 'Inception': Added 0 reviews.
Reviews for 'The Inception of Dramatic Representation': Added 0 reviews.
Reviews for 'The Inception of Seeking': Added 0 reviews.
Reviews for 'Bikini Inception': Added 0 reviews.
Reviews for 'Inception: The Cobol Job': Added 0 reviews.
Reviews for 'Inception: Music from the Motion Picture': Added 0 reviews.
Added these people to collection ('people'): 
Leonar

In [29]:
#Selects
results = query_col("Minecraft", "movies")
print(results)


[{'page_content': "Four misfits find themselves struggling with ordinary problems when they are suddenly pulled through a mysterious portal into the Overworld: a bizarre, cubic wonderland that thrives on imagination. To get back home, they'll have to master this world while embarking on a magical quest with an unexpected, expert crafter, Steve.", 'metadata': {'vote_average': 6.3, 'cast': 'Jason Momoa, Jack Black, Sebastian Eugene Hansen, Emma Myers, Danielle Brooks, Jennifer Coolidge, Rachel House, Allan Henry, Bram Scott-Breheny, Moana Williams, Jemaine Clement, Mark Wright, Yvette Parsons, Hiram Garcia, Bret McKenzie, Batanai Mashingaidze, Amanda Billing, Tommy Broadmore, Frankie Creagh-Leslie, Alison Quigan, John Smythe, Alex Tunui, Craig Mckinney, Joel Rindelaub, Antony Degreat, Victor Kazaroho, Ben Carpenter, Rowan Bacal, Brennan Standing, Dylan Chitekwe, Matt Berry, Alice May Connolly, Jens Bergensten, Daniel Middleton, Elizabeth Batty, Jessica Bravura, Kate McKinnon, ', 'vote_co